# 1. Replicate Simple recommender implementation

In [150]:
import pandas as pd
import numpy as np

In [151]:
base_df = pd.read_csv('data/IMBD.csv')

# delete dublicats of series
base_df = base_df.drop_duplicates(subset=['title'])
base_df = base_df.reset_index(drop=True)


display(base_df)

,title,year,certificate,duration,genre,rating,description,stars,votes
0,Cobra Kai,(2018– ),TV-14,30 min,"Action, Comedy, Drama",8.5,Decades after their 1984 All Valley Karate Tou...,"['Ralph Macchio, ', 'William Zabka, ', 'Courtn...","177,031"
1,The Crown,(2016– ),TV-MA,58 min,"Biography, Drama, History",8.7,Follows the political rivalries and romance of...,"['Claire Foy, ', 'Olivia Colman, ', 'Imelda St...","199,885"
2,Better Call Saul,(2015–2022),TV-MA,46 min,"Crime, Drama",8.9,The trials and tribulations of criminal lawyer...,"['Bob Odenkirk, ', 'Rhea Seehorn, ', 'Jonathan...","501,384"
3,Devil in Ohio,(2022),TV-MA,356 min,"Drama, Horror, Mystery",5.9,When a psychiatrist shelters a mysterious cult...,"['Emily Deschanel, ', 'Sam Jaeger, ', 'Gerardo...","9,773"
4,Cyberpunk: Edgerunners,(2022– ),TV-MA,24 min,"Animation, Action, Adventure",8.6,A Street Kid trying to survive in a technology...,"['Zach Aguilar, ', 'Kenichiro Ohashi, ', 'Emi ...","15,413"
...,...,...,...,...,...,...,...,...,...
7907,Alex,(2017–2019),NaN,47 min,"Action, Crime, Thriller",7.3,Add a Plot,"['Alain Darborg', '| ', ' Stars:', 'Dragomi...",33
7908,The Weekly with Wendy Mesley,(2018– ),NaN,NaN,"News, Reality-TV",NaN,How did Ontario's popular new Premier get so u...,"['Wendy Mesley, ', 'Billy Porter, ', 'Jacob To...",NaN
7909,The Drew Barrymore Show,(2020– ),TV-PG,44 min,Talk-Show,6.0,In the inaugural episode of The Drew Barrymore...,"['Adam Heydt', '| ', ' Stars:', 'Drew Barry...",19
7910,Hollywood Insider,(2018– ),NaN,NaN,Talk-Show,NaN,Behind the scenes of The Irishman.,"['Bobby Cannavale, ', 'Robert De Niro, ', 'Al ...",NaN


In [152]:
df = base_df[['title', 'rating', 'votes']]
display(df)

,title,rating,votes
0,Cobra Kai,8.5,"177,031"
1,The Crown,8.7,"199,885"
2,Better Call Saul,8.9,"501,384"
3,Devil in Ohio,5.9,"9,773"
4,Cyberpunk: Edgerunners,8.6,"15,413"
...,...,...,...
7907,Alex,7.3,33
7908,The Weekly with Wendy Mesley,NaN,NaN
7909,The Drew Barrymore Show,6.0,19
7910,Hollywood Insider,NaN,NaN


# Cheking null values in the dataset

In [153]:
df.isna().sum()

title        0
rating    1040
votes     1040
dtype: int64

In [154]:
df = df.dropna(subset=['votes'])
df.isna().sum()

title     0
rating    0
votes     0
dtype: int64

# Formula for calculating the weighted rating of a movie:

Weight formula:
\begin{equation} \text Weighted Rating (\bf WR) = \left({{\bf v} \over {\bf v} + {\bf m}} \cdot R\right) + \left({{\bf m} \over {\bf v} + {\bf m}} \cdot C\right) \end{equation}

In the above equation,

    v is the number of votes for the movie;

    m is the minimum votes required to be listed in the chart;

    R is the average rating of the movie;

    C is the mean vote across the whole report.


# Calculating value C

In [155]:
C = df['rating'].mean()
print(C)

6.5340948777648435


# Calculate the number of votes

In [156]:
df['votes'] = pd.to_numeric(df['votes'], errors='coerce')

m = df['votes'].quantile(0.90)
print(m)

809.0


In [157]:
test_df = df.copy().loc[df['votes'] >= m]
display(test_df)

,title,rating,votes
9,Blonde,6.2,935.0
17,1899,9.6,853.0
204,The Anthrax Attacks,6.2,865.0
485,Superwog,7.4,986.0
496,Lou,5.9,880.0
...,...,...,...
6976,The Untold Tales of Armistead Maupin,7.6,809.0
7027,That's It,6.8,892.0
7111,Craig Ferguson: I'm Here to Help,7.1,823.0
7234,Lucas Brothers: On Drugs,5.7,842.0


In [158]:
def weighted_rating(x, m=m, C=C):
    v = x['votes']
    R = x['rating']
    return (v/(v+m) * R) + (m/(m+v) * C)


In [159]:
test_df['score'] = test_df.apply(weighted_rating, axis=1)

test_df = test_df.sort_values('score', ascending=False)
test_df.head(20)

,title,rating,votes,score
17,1899,9.6,853.0,8.107631
2924,Yungnyong-i Nareusya,8.9,994.0,7.838426
7243,#HappyBirthdaySense8,9.0,844.0,7.793154
3531,Lastman,8.6,985.0,7.668385
4552,Daughters of Destiny,8.6,935.0,7.641676
3257,Puffin Rock,8.6,932.0,7.640025
4274,Shinchan,8.4,969.0,7.551003
4539,Like in the Movies,8.5,840.0,7.535526
2355,Ask the StoryBots,8.4,904.0,7.518787
2407,Stove League,8.3,918.0,7.472775


# 2. (optional) Replicate the the content based recommender implementation

In [160]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [161]:
display(base_df['description'])
df = base_df.copy()

0       Decades after their 1984 All Valley Karate Tou...
1       Follows the political rivalries and romance of...
2       The trials and tribulations of criminal lawyer...
3       When a psychiatrist shelters a mysterious cult...
4       A Street Kid trying to survive in a technology...
                              ...                        
7907                                           Add a Plot
7908    How did Ontario's popular new Premier get so u...
7909    In the inaugural episode of The Drew Barrymore...
7910                   Behind the scenes of The Irishman.
7911    Businessman Jerry Buss bets it all on the lack...
Name: description, Length: 7912, dtype: str

### Remove stop words like 'the', 'an', etc. since they do not give any useful information about the topic;

In [162]:
tfidf = TfidfVectorizer(stop_words='english')

### Replace not-a-number values with a blank string;

In [163]:
df['description'] = df['description'].fillna('')

### Creating TF-IDF matrix.

In [164]:
tfidf_matrix = tfidf.fit_transform(df['description'])

### Computing the cosine similarity matrix.

In [165]:
from sklearn.metrics.pairwise import linear_kernel

cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

### Create indexes

In [166]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
display(indices.head())

title
Cobra Kai                 0
The Crown                 1
Better Call Saul          2
Devil in Ohio             3
Cyberpunk: Edgerunners    4
dtype: int64

In [167]:
def get_recommendations(title, cosine_sim=cosine_sim):
    idx = indices[title]

    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    return df.iloc[movie_indices]['title']

In [168]:
get_recommendations('Cobra Kai')

7839    Karate World Champion Rates 11 Karate Scenes i...
4194                                    Absolute Dominion
6156                                  Golden Cane Warrior
5076                         Iron Fists and Kung Fu Kicks
565                                   Things Heard & Seen
4191                                  Bad Day for the Cut
2432                                        Word of Honor
405                                         Step Brothers
6220                                           Two Graves
307                                             Iron Fist
Name: title, dtype: str

In [169]:
get_recommendations('Supernatural')

3174          Super Giant Robot Brothers
3456             Oni: Thunder God's Tale
217                                Grimm
5280           Diary of a Night Watchman
2473                Twilight of the Gods
1457                            Spectral
6268              Cyborg 009 vs Devilman
1667                  Record of Ragnarok
4378    Jenni Rivera: Mariposa de Barrio
3570                           God Eater
Name: title, dtype: str